# COMP34812 NLI — Demo Notebook

**Group:** n  
**Category:** B — Deep Learning (non-transformer)  
**Model:** ResESIM + 1477d input embeddings

| Component | Dim | Reference |
|---|---|---|
| GloVe | 300 | Pennington et al., 2014 |
| ELMo | 1024 | Peters et al., 2018 |
| CharCNN | 100 | Kim et al., 2016 |
| POS embeddings | 50 | Universal Dependencies |
| Negation flags | 3 | spaCy dep parse + boundary |
| **Total** | **1477** | |

## Instructions

1. Run all cells top to bottom
2. Predictions saved to `Group_n_B.csv`

**Input CSV:** must have `premise` and `hypothesis` columns

```
test.csv
    ↓  ELMo pre-computation  (~5 mins, A100)
    ↓  GloVe + CharCNN + POS + Negation
    ↓  inference_embeddings.npz  [N, 64, 1477]
    ↓  OracleNet (ResESIM)
    ↓  Group_n_B.csv
```

## 1. Environment Setup

**Skip this cell if running locally** — only needed on Colab.

## 2. Setup Path + Verify Files

In [21]:
import sys
sys.path.append('.')
!{sys.executable} -m pip install -q spacy
!{sys.executable} -m spacy download en_core_web_sm
!{sys.executable} -m pip install scikit-learn
print('spaCy model ready.')
print('Repo ready.')




[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
  Using cached scikit_learn-1.7.2-cp310-cp310-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached scipy-1.15.3-cp310-cp310-macosx_14_0_arm64.whl.metadata (61 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.7.2-cp310-cp310-macosx_12_0_arm64.whl (8.7 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached s

In [10]:
# ──required files ─────────────────────────────────────────────────────
import os

required = {
    './notebook_data/meta.pt': 'Vocab + GloVe meta',
    './final_model_versions/ff2f02d4/best_model.pt': 'Trained model weights',
    './data/NLI_trial.csv': 'Trial data for inference',
    './notebook_data/elmo_model/options.json': 'ELMo options',
    './notebook_data/elmo_model/weights.hdf5': 'ELMo weights',
}

all_ok = True
for path, desc in required.items():
    exists = os.path.exists(path)
    size   = f'  ({os.path.getsize(path)/1e6:.1f} MB)' if exists else ''
    status = '✓' if exists else '✗  MISSING'
    print(f'  {status}  {path}{size}')
    if not exists: all_ok = False

print()
print('All files present — continue.' if all_ok else 'Fix missing files before continuing.')

  ✓  ./notebook_data/meta.pt  (1.0 MB)
  ✓  ./final_model_versions/ff2f02d4/best_model.pt  (86.9 MB)
  ✓  ./data/NLI_trial.csv  (0.0 MB)
  ✓  ./notebook_data/elmo_model/options.json  (0.0 MB)
  ✓  ./notebook_data/elmo_model/weights.hdf5  (374.4 MB)

All files present — continue.


## 3. Pre-compute Embeddings

Uses `EmbeddingPrecomputer` from `precomputeClasses.py`.  
Steps run automatically:
1. Load meta (vocab, char2idx, pos2idx, GloVe)
2. Build embedding layers (CharCNN, POS, GloVe)
3. ELMo pre-computation via Python 3.10 venv
4. Full 1477d pre-computation
5. Save `inference_embeddings.npz`

In [11]:
from precomputeClasses import EmbeddingPrecomputer

pc = EmbeddingPrecomputer(
    meta_path    = './notebook_data/meta.pt',
    elmo_options = './notebook_data/elmo_model/options.json',
    elmo_weights = './notebook_data/elmo_model/weights.hdf5',
    elmo_venv    = './venv310/bin/python',
)

pc.run(
    csv_path   = './data/NLI_trial.csv',
    output_npz = './notebook_data/inference_embeddings.npz',
)


  EmbeddingPrecomputer — COMP34812 NLI
  Input:  ./data/NLI_trial.csv
  Output: ./notebook_data/inference_embeddings.npz
  Dim:    1477  (GloVe+ELMo+CharCNN+POS+Neg)

[1/5] Loading meta from notebook_data/meta.pt...
      vocab=566  char=256  pos=18
[2/5] Building embedding layers...
      GloVe frozen  |  CharCNN  |  POS  |  spaCy en_core_web_sm
[3/5] Pre-computing ELMo (Peters et al., 2018)...


/Users/kaanoktem/NLU-CW/venv310/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/Users/kaanoktem/NLU-CW/venv310/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: dlopen(/Users/kaanoktem/NLU-CW/venv310/lib/python3.10/site-packages/torchvision/image.so, 0x0006): Symbol not found: __ZN2at4_ops19empty_memory_format4callEN3c108ArrayRefIxEENS2_8optionalINS2_10ScalarTypeEEENS5_INS2_6LayoutEEENS5_INS2_6DeviceEEENS5_IbEENS5_INS2_12MemoryFormatEEE
  Referenced from: <06D2C3BD-26E5-3AB9-A866-63839BE393A7> /Users/kaanoktem/NLU-CW/venv310/lib/python3.10/site-packages/torc

ELMo on cpu
  50 premises...
    50/50
  50 hypotheses...
    50/50
ELMo done.
      ELMo saved.
[4/5] Pre-computing 1477d embeddings...


  batches: 100%|██████████| 1/1 [00:01<00:00,  1.30s/it]


[5/5] Saving ./notebook_data/inference_embeddings.npz...
      Saved: 0.01 GB  |  shape (50, 64, 1477)
      NaN check: premise=0 ✓  hypothesis=0 ✓

✓ Pre-computation complete — ./notebook_data/inference_embeddings.npz


'./notebook_data/inference_embeddings.npz'

## 4. Load Model + Run Inference

In [12]:
# ── Select device ─────────────────────────────────────────────────────────────
import torch

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'Device: {device}')

Device: mps


In [13]:
# ── Load dataset and dataloader ────────────────────────────────────────────────
from res_esim.loader.res_esim_dataset import ResESIM_Dataset
from torch.utils.data import DataLoader

dataset = ResESIM_Dataset('./notebook_data/inference_embeddings.npz')
loader  = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0)

Loading ./notebook_data/inference_embeddings.npz...
  50 samples, dim=1477


In [14]:
# ── Load OracleNet model ──────────────────────────────────────────────────────
from res_esim.model_layers.oracle_net import OracleNet

model = OracleNet(
    input_dim=1477,
    hidden_dim=512,
    num_blocks=3,
    num_classes=2,
    dropout_rate=0.2,
    num_heads=8,
)
model.load_state_dict(torch.load(
    './final_model_versions/ff2f02d4/best_model.pt',
    map_location=device,
    weights_only=False,
))
model.to(device).eval()
print('Model loaded and set to eval mode.')

Model loaded and set to eval mode.


In [15]:
# ── Run inference ─────────────────────────────────────────────────────────────
from tqdm import tqdm

all_preds = []
with torch.no_grad():
    for batch in tqdm(loader, desc='inference'):
        logits = model(
            batch['premise_embedding'].to(device),
            batch['hyp_embedding'].to(device),
            batch['premise_length'].to(device),
            batch['hyp_length'].to(device),
        )
        all_preds.extend(logits.argmax(dim=-1).cpu().tolist())

print(f'Inference complete: {len(all_preds)} predictions')

inference: 100%|██████████| 2/2 [00:00<00:00,  6.51it/s]

Inference complete: 50 predictions


In [16]:
# ── Save predictions ──────────────────────────────────────────────────────────
import pandas as pd

test_df = pd.read_csv('./data/NLI_trial.csv')
test_df['label'] = all_preds
test_df[['premise', 'hypothesis', 'label']].to_csv('Group_n_B.csv', index=False)

print(f'Saved Group_n_B.csv  ({len(all_preds)} predictions)')
print(f'Label distribution: {pd.Series(all_preds).value_counts().sort_index().to_dict()}')

Saved Group_n_B.csv  (50 predictions)
Label distribution: {0: 17, 1: 33}


In [22]:
# ── Evaluate F1 (only if ground-truth labels are available) ───────────────────
import pandas as pd
from sklearn.metrics import f1_score, classification_report

truth = pd.read_csv('./data/NLI_trial.csv')

if 'label' in truth.columns:
    y_true = truth['label'].tolist()
    y_pred = all_preds
    f1 = f1_score(y_true, y_pred, average='macro')
    print(f'Macro F1: {f1:.4f}\n')
    print(classification_report(y_true, y_pred, target_names=['Not Entailment (0)', 'Entailment (1)']))
else:
    print('No ground-truth labels available — skipping F1 evaluation.')

Macro F1: 0.7159

                    precision    recall  f1-score   support

Not Entailment (0)       0.88      0.56      0.68        27
    Entailment (1)       0.64      0.91      0.75        23

          accuracy                           0.72        50
         macro avg       0.76      0.73      0.72        50
      weighted avg       0.77      0.72      0.71        50



## Done

Predictions saved to `Group_n_B.csv`  
Download and submit via Canvas.